# Numerical Optimization: Four Classical Methods
### Golden Section Search · Bisection Method · Fibonacci Search · Steepest Ascent

**Authors:** Debdan Samanta (24IM10034), Musaib Bin Bashir (24IM10047), Srijan Ghosh (24IM10062)
**Course:** Operations Research II  

---
Each section contains: (1) a brief method description, (2) a self-contained implementation, (3) a dedicated test case on a **different** example function, (4) an iteration table, and (5) convergence plots.

In [ ]:
#    Shared imports                                                             
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})
print('Libraries loaded.')

---
## Method 1 - Golden Section Search

### Description
The Golden Section Search reduces a unimodal search interval by the golden ratio
$\varphi = (\sqrt{5}-1)/2 \approx 0.618$ at each step. Two interior points are placed:
$$x_1 = b - \varphi(b-a), \qquad x_2 = a + \varphi(b-a)$$
Comparing $f(x_1)$ and $f(x_2)$ eliminates one sub-interval. One function evaluation is
reused per iteration, giving 38.2% reduction of the bracket per step.

### Test Case
$$\text{Minimize}\quad f(x) = x^2 - 4x + 7 \quad\text{on}\quad [0,5]$$
Analytical minimum: $x^* = 2$, $f^* = 3$.

In [ ]:

# METHOD 1  - GOLDEN SECTION SEARCH
# Test function : f(x) = x^2 - 4x + 7  on [0, 5]
# Analytical    : x* = 2,  f* = 3


def golden_section_search(f, a, b, tol=1e-6, max_iter=200):
    """
    Golden Section Search  - minimize a unimodal function f on [a, b].

    Parameters
    ----------
    f        : callable  Unimodal objective function.
    a, b     : float     Initial search interval.
    tol      : float     Stop when (b - a) < tol.
    max_iter : int       Maximum iterations.

    Returns
    -------
    x_opt   : float        Approximate minimizer.
    f_opt   : float        Approximate minimum value.
    history : list[dict]   Per-iteration record.
    """
    phi = (np.sqrt(5) - 1) / 2          # golden ratio ≈ 0.61803
    history = []

    x1 = b - phi * (b - a)
    x2 = a + phi * (b - a)
    f1, f2 = f(x1), f(x2)

    for k in range(1, max_iter + 1):
        history.append(dict(Iter=k, a=a, b=b, x1=x1, x2=x2,
                            f_x1=f1, f_x2=f2, width=b - a))
        if b - a < tol:
            break
        if f1 < f2:               # minimum lies in [a, x2]
            b, x2, f2 = x2, x1, f1
            x1 = b - phi * (b - a)
            f1 = f(x1)
        else:                     # minimum lies in [x1, b]
            a, x1, f1 = x1, x2, f2
            x2 = a + phi * (b - a)
            f2 = f(x2)

    x_opt = (a + b) / 2
    return x_opt, f(x_opt), history


#  Run test case 
f_gs  = lambda x: x**2 - 4*x + 7
x_gs, fval_gs, hist_gs = golden_section_search(f_gs, a=0.0, b=5.0, tol=1e-6)

print('Golden Section Search  - Test Case')
print(f'  f(x) = x² - 4x + 7  on [0, 5]')
print(f'  Analytical : x* = 2.000000   f* = 3.000000')
print(f'  Numerical  : x* = {x_gs:.6f}   f* = {fval_gs:.6f}')
print(f'  Iterations : {len(hist_gs)}')

In [ ]:
#  Iteration table 
df_gs = pd.DataFrame(hist_gs)[['Iter','a','b','x1','x2','f_x1','f_x2','width']]
df_gs.columns = ['Iter','a','b','x1','x2','f(x1)','f(x2)','Width']
display(
    df_gs.head(12).style
    .format({'Iter': '{:.0f}', **{c: '{:.6f}' for c in df_gs.columns if c != 'Iter'}})
    .set_caption('Table 1  - Golden Section Search: First 12 Iterations')
    .highlight_min(subset=['Width'], color='#d4f7dc')
)

In [ ]:
#    Plots  
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

xs = np.linspace(0, 5, 500)
ax1.plot(xs, f_gs(xs), color='steelblue', lw=2.2, label='f(x) = x² − 4x + 7')
cols = plt.cm.YlOrRd(np.linspace(0.3, 0.85, min(8, len(hist_gs))))
for row, c in zip(hist_gs[:8], cols):
    ax1.axvspan(row['a'], row['b'], alpha=0.18, color=c)
ax1.axvline(x_gs, color='crimson', lw=1.6, ls='--', label=f'x* ≈ {x_gs:.4f}')
ax1.scatter([x_gs], [fval_gs], color='crimson', s=70, zorder=6)
ax1.set(xlabel='x', ylabel='f(x)', title='Figure 1a  - Interval Narrowing')
ax1.legend(fontsize=9)

widths = [r['width'] for r in hist_gs]
ax2.semilogy(range(1, len(widths)+1), widths, 'o-', color='steelblue', ms=4, lw=1.5)
ax2.set(xlabel='Iteration', ylabel='Interval width (log scale)',
        title='Figure 1b  - Bracket Width Convergence')
plt.tight_layout()
plt.savefig('fig1_golden_section.png', bbox_inches='tight')
plt.show()

---
## Method 2  - Bisection Method

### Description
The Bisection Method finds a zero of $f'(x)$ (a stationary point of $f$) by bracketing and
halving. Given $[a,b]$ with $f'(a)\cdot f'(b)<0$, the midpoint $m=(a+b)/2$ is evaluated:

- If $f'(a)\cdot f'(m) < 0$: root in $[a,m]$, set $b = m$
- Otherwise: root in $[m,b]$, set $a = m$

The bracket width halves exactly each iteration  - linear convergence with rate $\frac{1}{2}$.

### Test Case
$$\text{Minimize}\quad f(x) = e^x - 5x \quad\text{on}\quad [0,4]$$
$$f'(x) = e^x - 5 = 0 \implies x^* = \ln 5 \approx 1.60944, \quad f^* \approx -3.04719$$

In [ ]:

# METHOD 2  - BISECTION METHOD
# Test function : f(x) = e^x - 5x  on [0, 4]
# Analytical    : x* = ln(5) ≈ 1.60944,  f* ≈ -3.04719


def bisection_method(df, a, b, tol=1e-6, max_iter=200):
    """
    Bisection Method  - find a zero of the derivative df in [a, b].

    Parameters
    ----------
    df       : callable  Derivative f'(x) of the objective.
    a, b     : float     Bracket satisfying df(a)*df(b) < 0.
    tol      : float     Stop when (b - a) < tol.
    max_iter : int       Maximum iterations.

    Returns
    -------
    x_opt   : float
    history : list[dict]
    """
    assert df(a) * df(b) < 0, "df must have opposite signs at a and b."
    history = []

    for k in range(1, max_iter + 1):
        mid = (a + b) / 2
        dfm = df(mid)
        history.append(dict(Iter=k, a=a, b=b, mid=mid,
                            df_mid=dfm, width=b - a))
        if b - a < tol:
            break
        if df(a) * dfm < 0:   # root in left half
            b = mid
        else:                  # root in right half
            a = mid

    return (a + b) / 2, history


#    Run test case                                                             ─
f_bs  = lambda x: np.exp(x) - 5*x
df_bs = lambda x: np.exp(x) - 5

x_bs, hist_bs = bisection_method(df_bs, a=0.0, b=4.0, tol=1e-6)

print('Bisection Method  - Test Case')
print(f'  f(x) = eˣ - 5x  on [0, 4]')
print(f'  Analytical : x* = {np.log(5):.6f}   f* = {f_bs(np.log(5)):.6f}')
print(f'  Numerical  : x* = {x_bs:.6f}   f* = {f_bs(x_bs):.6f}')
print(f'  Iterations : {len(hist_bs)}')

In [ ]:
#    Iteration table                                                           ─
df_tbl = pd.DataFrame(hist_bs)[['Iter','a','b','mid','df_mid','width']]
df_tbl.columns = ['Iter','a','b','midpoint',"f'(mid)",'Width']
display(
    df_tbl.head(12).style
    .format({'Iter': '{:.0f}', **{c: '{:.6f}' for c in df_tbl.columns if c != 'Iter'}})
    .set_caption("Table 2  - Bisection Method: First 12 Iterations")
    .highlight_min(subset=['Width'], color='#d4f7dc')
)

In [ ]:
#    Plots                                                                     ─
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

xs = np.linspace(0, 4, 500)
ax1.plot(xs, f_bs(xs),  color='steelblue',  lw=2.2, label='f(x) = eˣ − 5x')
ax1.plot(xs, df_bs(xs), color='darkorange', lw=2.0, ls='--', label="f'(x) = eˣ − 5")
ax1.axhline(0, color='grey', lw=0.8)
ax1.axvline(x_bs, color='crimson', lw=1.6, ls=':', label=f'x* ≈ {x_bs:.5f}')
ax1.scatter([x_bs], [f_bs(x_bs)], color='crimson', s=70, zorder=6)
ax1.set(xlabel='x', ylabel='Value', title="Figure 2a  - f(x) and f'(x)")
ax1.legend(fontsize=9)

dfmids = [abs(r['df_mid']) for r in hist_bs]
ax2.semilogy(range(1, len(dfmids)+1), dfmids, 's-', color='darkorange', ms=4, lw=1.5)
ax2.set(xlabel='Iteration', ylabel="|f'(mid)| (log scale)",
        title="Figure 2b  - |f'(midpoint)| Convergence")
plt.tight_layout()
plt.savefig('fig2_bisection.png', bbox_inches='tight')
plt.show()

---
## Method 3  - Fibonacci Search

### Description
Fibonacci Search positions its two interior points using Fibonacci numbers
$F_1=F_2=1,\; F_k = F_{k-1}+F_{k-2}$.
For $n$ iterations the reduction ratio at step $k$ is $F_{n-k}/F_{n-k+2}$,
giving total interval reduction $1/F_{n+2}$  - **provably optimal** for a fixed evaluation budget.

$$x_1 = a + \frac{F_n}{F_{n+2}}(b-a), \qquad x_2 = a + \frac{F_{n+1}}{F_{n+2}}(b-a)$$

### Test Case
$$\text{Minimize}\quad f(x) = (x-1)^4 + (x-1)^2 \quad\text{on}\quad [-2,4]$$
Analytical minimum: $x^* = 1$, $f^* = 0$.

In [ ]:
#  
# METHOD 3  - FIBONACCI SEARCH
# Test function : f(x) = (x-1)^4 + (x-1)^2  on [-2, 4]
# Analytical    : x* = 1,  f* = 0
#  

def fibonacci_search(f, a, b, n=18):
    """
    Fibonacci Search  - minimize a unimodal function f on [a, b] using n steps.

    Parameters
    ----------
    f    : callable  Unimodal objective function.
    a, b : float     Search interval.
    n    : int       Number of reduction steps; uses F_{n+2} evaluations total.

    Returns
    -------
    x_opt   : float
    f_opt   : float
    history : list[dict]
    """
    # Generate F[0] .. F[n+2]
    F = [1, 1]
    while len(F) < n + 3:
        F.append(F[-1] + F[-2])

    history = []
    L  = b - a
    x1 = a + F[n]   / F[n+2] * L
    x2 = a + F[n+1] / F[n+2] * L
    f1, f2 = f(x1), f(x2)

    for k in range(1, n + 1):
        history.append(dict(k=k, a=a, b=b, x1=x1, x2=x2,
                            f_x1=f1, f_x2=f2, width=b - a))
        ni = n - k
        if f1 > f2:            # minimum in [x1, b]
            a  = x1
            x1, f1 = x2, f2
            d = F[ni+2] if ni+2 < len(F) else F[-1]
            r = F[ni+1] if ni+1 < len(F) else F[-1]
            x2 = a + r / d * (b - a)
            f2 = f(x2)
        else:                  # minimum in [a, x2]
            b  = x2
            x2, f2 = x1, f1
            d = F[ni+2] if ni+2 < len(F) else F[-1]
            r = F[ni]   if ni   < len(F) else F[-1]
            x1 = a + r / d * (b - a)
            f1 = f(x1)

    x_opt = (a + b) / 2
    return x_opt, f(x_opt), history


#    Run test case                                                             ─
f_fb = lambda x: (x - 1)**4 + (x - 1)**2
x_fb, fval_fb, hist_fb = fibonacci_search(f_fb, a=-2.0, b=4.0, n=18)

print('Fibonacci Search  - Test Case')
print(f'  f(x) = (x-1)⁴ + (x-1)²  on [-2, 4]')
print(f'  Analytical : x* = 1.000000   f* = 0.000000')
print(f'  Numerical  : x* = {x_fb:.6f}   f* = {fval_fb:.6f}')
print(f'  Iterations : {len(hist_fb)}')
print(f'  Final width: {hist_fb[-1]["width"]:.2e}')

In [ ]:
#    Iteration table                                                           ─
df_fb = pd.DataFrame(hist_fb)[['k','a','b','x1','x2','f_x1','f_x2','width']]
df_fb.columns = ['k','a','b','x1','x2','f(x1)','f(x2)','Width']
display(
    df_fb.head(12).style
    .format({'k': '{:.0f}', **{c: '{:.6f}' for c in df_fb.columns if c != 'k'}})
    .set_caption('Table 3  - Fibonacci Search: First 12 Iterations')
    .highlight_min(subset=['Width'], color='#d4f7dc')
)

In [ ]:
#    Plots                                                                     ─
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

xs = np.linspace(-2, 4, 500)
ax1.plot(xs, f_fb(xs), color='purple', lw=2.2,
         label='f(x) = (x−1)⁴ + (x−1)²')
cols_fb = plt.cm.Purples(np.linspace(0.35, 0.85, min(8, len(hist_fb))))
for row, c in zip(hist_fb[:8], cols_fb):
    ax1.axvspan(row['a'], row['b'], alpha=0.15, color=c)
ax1.axvline(x_fb, color='crimson', lw=1.6, ls='--', label=f'x* ≈ {x_fb:.5f}')
ax1.scatter([x_fb], [fval_fb], color='crimson', s=70, zorder=6)
ax1.set(xlabel='x', ylabel='f(x)', title='Figure 3a  - Interval Narrowing')
ax1.legend(fontsize=9)

widths_fb = [r['width'] for r in hist_fb]
ax2.semilogy(range(1, len(widths_fb)+1), widths_fb,
             'D-', color='purple', ms=4, lw=1.5)
ax2.set(xlabel='Iteration k', ylabel='Interval width (log scale)',
        title='Figure 3b  - Bracket Width Convergence')
plt.tight_layout()
plt.savefig('fig3_fibonacci.png', bbox_inches='tight')
plt.show()

---
## Method 4  - Steepest Ascent (Gradient Ascent)

### Description
Steepest Ascent **maximizes** $f(\mathbf{x})$ by iteratively stepping in the gradient direction:
$$\mathbf{x}_{k+1} = \mathbf{x}_k + \alpha\,\nabla f(\mathbf{x}_k), \qquad \alpha > 0$$
The algorithm stops when $\|\nabla f(\mathbf{x}_k)\| < \text{tol}$.
Choosing $\alpha$ too large causes divergence; too small slows convergence.
For minimization, replace $+\alpha$ with $-\alpha$ (gradient descent).

### Test Case
$$\text{Maximize}\quad f(x,y) = -(x-3)^2 - 2(y-2)^2 + 10$$
$$\nabla f = \bigl(-2(x-3),\;-4(y-2)\bigr), \quad (x^*,y^*) = (3,2), \quad f^* = 10$$

In [ ]:
#  
# METHOD 4  - STEEPEST ASCENT (GRADIENT ASCENT)
# Test function : f(x,y) = -(x-3)^2 - 2(y-2)^2 + 10
# Analytical    : (x*,y*) = (3, 2),  f* = 10
#  

def steepest_ascent(f, grad_f, x0, alpha=0.25, tol=1e-8, max_iter=300):
    """
    Steepest Ascent  - maximize f by following the gradient.

    Parameters
    ----------
    f        : callable   Objective to MAXIMIZE. f(x) where x is ndarray.
    grad_f   : callable   Gradient. Returns ndarray of same shape as x.
    x0       : array-like Starting point.
    alpha    : float      Fixed step size (learning rate).
    tol      : float      Stop when ||grad_f(x)|| < tol.
    max_iter : int        Maximum iterations.

    Returns
    -------
    x_opt   : ndarray
    f_opt   : float
    history : list[dict]
    """
    x = np.array(x0, dtype=float)
    history = []

    for k in range(max_iter):
        g     = np.asarray(grad_f(x), dtype=float)
        gnorm = np.linalg.norm(g)
        history.append(dict(Iter=k, x=x[0], y=x[1],
                            f_xy=f(x), grad_norm=gnorm,
                            gx=g[0], gy=g[1]))
        if gnorm < tol:
            break
        x = x + alpha * g        # ascent step

    return x, f(x), history


#    Run test case                                                             ─
f_sa    = lambda xy: -(xy[0]-3)**2 - 2*(xy[1]-2)**2 + 10
grad_sa = lambda xy: np.array([-2*(xy[0]-3), -4*(xy[1]-2)])
x0_sa   = np.array([-1.0, -1.0])        # start far from optimum

xy_sa, fval_sa, hist_sa = steepest_ascent(
    f_sa, grad_sa, x0_sa, alpha=0.25, tol=1e-8)

print('Steepest Ascent  - Test Case')
print(f'  f(x,y) = -(x-3)² - 2(y-2)² + 10')
print(f'  Start      : x₀ = (-1, -1)')
print(f'  Analytical : (x*,y*) = (3, 2)   f* = 10')
print(f'  Numerical  : (x*,y*) = ({xy_sa[0]:.6f}, {xy_sa[1]:.6f})   f* = {fval_sa:.6f}')
print(f'  Iterations : {len(hist_sa)}')

In [ ]:
#    Iteration table                                                           ─
df_sa = pd.DataFrame(hist_sa)[['Iter','x','y','f_xy','grad_norm','gx','gy']]
df_sa.columns = ['Iter','x','y','f(x,y)','||grad||','df/dx','df/dy']
display(
    df_sa.head(12).style
    .format({'Iter': '{:.0f}', **{c: '{:.6f}' for c in df_sa.columns if c != 'Iter'}})
    .set_caption('Table 4  - Steepest Ascent: First 12 Iterations')
    .highlight_max(subset=['f(x,y)'], color='#d4f7dc')
    .highlight_min(subset=['||grad||'], color='#fff3cd')
)

In [ ]:
#    Plots                                                                     ─
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

gx = np.linspace(-2, 6, 300)
gy = np.linspace(-2, 5, 300)
X, Y = np.meshgrid(gx, gy)
Z = -(X-3)**2 - 2*(Y-2)**2 + 10

cp = ax1.contourf(X, Y, Z, levels=20, cmap='RdYlGn', alpha=0.75)
ax1.contour(X, Y, Z, levels=20, colors='white', linewidths=0.4, alpha=0.4)
plt.colorbar(cp, ax=ax1, label='f(x, y)')

px = [r['x'] for r in hist_sa]
py = [r['y'] for r in hist_sa]
ax1.plot(px, py, 'o-', color='navy', ms=3, lw=1.5, label='Ascent path')
ax1.plot(px[0], py[0], 'rs', ms=10, label='Start (−1, −1)')
ax1.plot(3, 2, 'w*', ms=16, markeredgecolor='navy', markeredgewidth=0.8,
         label='Maximum (3, 2)')
ax1.set(xlabel='x', ylabel='y', title='Figure 4a  - Gradient Ascent Path')
ax1.legend(fontsize=8, loc='lower right')

iters  = [r['Iter']     for r in hist_sa]
fvals  = [r['f_xy']     for r in hist_sa]
gnorms = [r['grad_norm'] for r in hist_sa]

ax2.plot(iters, fvals, 'g-o', ms=3, lw=1.5, label='f(x,y)')
ax2.set_ylabel('f(x, y)', color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax3 = ax2.twinx()
ax3.semilogy(iters, gnorms, 'b--s', ms=3, lw=1.5, label='||grad||')
ax3.set_ylabel('||grad|| (log scale)', color='navy')
ax3.tick_params(axis='y', labelcolor='navy')
ax2.set(xlabel='Iteration', title='Figure 4b  - f value and Gradient Norm')
lines = ax2.get_lines() + ax3.get_lines()
ax2.legend(lines, [l.get_label() for l in lines], fontsize=9, loc='center right')

plt.tight_layout()
plt.savefig('fig4_steepest_ascent.png', bbox_inches='tight')
plt.show()

---
## Summary of Test Cases

| Method | Test Function | Interval / Start | Analytical $x^*$ | $f^*$ |
|---|---|---|---|---|
| Golden Section | $x^2 - 4x + 7$ | $[0, 5]$ | $x^* = 2$ | $3$ |
| Bisection | $e^x - 5x$ | $[0, 4]$ | $x^* = \ln 5 \approx 1.6094$ | $-3.0472$ |
| Fibonacci ($n=18$) | $(x-1)^4 + (x-1)^2$ | $[-2, 4]$ | $x^* = 1$ | $0$ |
| Steepest Ascent | $-(x-3)^2-2(y-2)^2+10$ | $(-1,-1)$ | $(x^*,y^*)=(3,2)$ | $10$ |

---
## Adapting to Your Own Function

```python
# Golden Section  - any unimodal 1-D function, no derivative needed
my_f = lambda x: np.sin(x) + 0.3*x
x_opt, f_opt, history = golden_section_search(my_f, a=2, b=6, tol=1e-8)

# Bisection  - supply the derivative; ensure df(a)*df(b) < 0
my_df = lambda x: np.cos(x) + 0.3
x_opt, history = bisection_method(my_df, a=2, b=5, tol=1e-8)

# Fibonacci  - any unimodal 1-D function; raise n for more precision
my_f = lambda x: (x - 2.5)**2 + np.cos(x)
x_opt, f_opt, history = fibonacci_search(my_f, a=0, b=5, n=25)

# Steepest Ascent  - any smooth n-D function; negate for minimization
my_f    = lambda xy: -(xy[0]-1)**2 - (xy[1]+2)**2 + 5
my_grad = lambda xy: np.array([-2*(xy[0]-1), -2*(xy[1]+2)])
xy_opt, f_opt, history = steepest_ascent(my_f, my_grad, x0=[4, 4], alpha=0.3)
```